# Exercise 1 - KNN on MNIST dataset

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [2]:
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X, y = mnist.data, mnist.target.astype(int)

X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

knn = KNeighborsClassifier()
param_grid = {"n_neighbors":[3,4,5], "weights":["uniform","distance"]}

grid = GridSearchCV(knn, param_grid, cv=3, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)

best_knn = grid.best_estimator_
y_pred = best_knn.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))

Best parameters: {'n_neighbors': 3, 'weights': 'uniform'}
Best CV accuracy: nan
Test accuracy: 0.9705


# Exercise 2 - KNN on augmented MNIST dataset

In [3]:
def shift_images(X, dx, dy):
    images = X.reshape(-1, 28, 28)
    shifted = np.roll(images, shift=(dy, dx), axis=(1, 2))

    if dy > 0:
        shifted[:, :dy, :] = 0
    elif dy < 0:
        shifted[:, dy:, :] = 0

    if dx > 0:
        shifted[:, :, :dx] = 0
    elif dx < 0:
        shifted[:, :, dx:] = 0

    return shifted.reshape(-1, 784)

X_left  = shift_images(X_train, -1, 0)
X_right = shift_images(X_train, 1, 0)
X_up    = shift_images(X_train, 0, -1)
X_down  = shift_images(X_train, 0, 1)

X_train_aug = np.concatenate([X_train, X_left, X_right, X_up, X_down])
y_train_aug = np.tile(y_train, 5)

print("Original:", X_train.shape)
print("Augmented:", X_train_aug.shape)

Original: (60000, 784)
Augmented: (300000, 784)


In [4]:
best_knn.fit(X_train_aug, y_train_aug)
y_pred_aug = best_knn.predict(X_test)
print("Test accuracy after augmentation:", accuracy_score(y_test, y_pred_aug))

Test accuracy after augmentation: 0.9756


The accuracy slightly increases for the dataset with data augmented/training set explanded (4 extra images per image with each image shifted by one pixel in each direction).